In [3]:
import re
import os
import re 
import time
import warnings

from IPython.display import clear_output

import jsonlines
import polars as pl

warnings.filterwarnings('ignore')

from orqa.utils import sanitize_string, get_resource_metadata

In [ ]:
tag = 'UK'
from_ = 0
to_ = 55556

data_path       = f'../data'
tables_path     = f'{data_path}/datasets/{tag}/tables/tables_from{from_}_to{to_}'
metadata_path   = f'{data_path}/datasets/{tag}/metadata/metadata_from{from_}_to{to_}.jsonl'
log_path        = f'{data_path}/log/{tag}/3_join_evaluation_{time.strftime('%y%m%d_%H_%M_%S')}.log'

evaluated_path  = f'{data_path}/outputs/{tag}/evaluated_joins.csv'

UP_TO_ROW           = 100

MAX_N_COLUMNS       = 10

# to limit the context passed to the LLM-agent (the "notes" field may be very long)
MAX_LENGTH_NOTES    = 300

# number of sampled rows passed to the LLM into the question context
N_ROWS_SAMPLE       = 3

# number of values in common between the candidate joinable columnes
# passed to the LLM into the question context
MAX_COMM_CELLS      = 10

MIN_SCORE           = 0
MAX_SCORE           = 10

table_ids = list(sorted(os.listdir(tables_path), reverse=True))

with jsonlines.open(metadata_path) as fr:
    metadata = {rsc['id']: md for md in fr.iter() for rsc in md['resources'] if rsc['format'] == 'CSV'}
evaluated_joins = pl.read_csv(evaluated_path, ignore_errors=True)

evaluations = []
score = -1

for i, row in enumerate(evaluated_joins.sample(UP_TO_ROW).rows()):
    try:
        # take relevant information for the llm-agent
        r_tab_id, s_tab_id, _, _, original_r_col_name, original_s_col_name, r_pkg_id, s_pkg_id, score, _ = row

        _, r_rsc_name, r_pkg_id, pkg_title, r_pkg_notes, r_pkg_keywords, r_pkg_tags = get_resource_metadata(r_tab_id, table_ids, metadata)
        _, s_rsc_name, s_pkg_id, pkg_title, s_pkg_notes, s_pkg_keywords, s_pkg_tags = get_resource_metadata(s_tab_id, table_ids, metadata)

        # limit the length of the notes and remove some chars (needed?)
        r_pkg_notes = re.sub(r"(\n|\r|\t)", " ", r_pkg_notes)[:MAX_LENGTH_NOTES]
        s_pkg_notes = re.sub(r"(\n|\r|\t)", " ", s_pkg_notes)[:MAX_LENGTH_NOTES]

        r_col_name, s_col_name = sanitize_string(original_r_col_name), sanitize_string(original_s_col_name)
        
        # read the tables 
        r_df = (
            pl
            .scan_parquet(f'{tables_path}/{table_ids[r_tab_id]}')
            .select(
                pl.all().map_elements(sanitize_string, pl.String)
            )
            .rename(sanitize_string)
            .collect()                
        )

        s_df = (
            pl
            .scan_parquet(f'{tables_path}/{table_ids[s_tab_id]}')
            .select(
                pl.all().map_elements(sanitize_string, pl.String)
            )
            .rename(sanitize_string)
            .collect()                
        )

        # dtype conversion to int/float
        for column in r_df.columns:
            try:
                dtype = pl.Float32 if any(',' in str(x) or '.' in str(x) for x in set(r_df.select(column).sample(100, with_replacement=True).to_series())) else pl.Int32
                r_df = r_df.with_columns(pl.col(column).cast(dtype))
            except:
                continue

        for column in s_df.columns:
            try:
                dtype = pl.Float32 if any(',' in str(x) or '.' in str(x) for x in set(s_df.select(column).sample(100, with_replacement=True).to_series())) else pl.Int32
                s_df = s_df.with_columns(pl.col(column).cast(dtype))
            except:
                continue
        
        # for the evaluation keep only the first MAX_N_COLUMNS columns 
        r_df = r_df.drop(r_col_name).insert_column(0, r_df.get_column(r_col_name))
        s_df = s_df.drop(s_col_name).insert_column(0, s_df.get_column(s_col_name))
                    
        s_df.get_column(s_col_name)
        common_cells = list(set(r_df.get_column(r_col_name)) & set(s_df.get_column(s_col_name)))
        common_cells = common_cells[:MAX_COMM_CELLS]                        
    except:
        continue

    # pass a formatted version of the tables to the completion client,
    # only a small portion of the tables
    with pl.Config(
        tbl_hide_dataframe_shape=True,
        tbl_width_chars=1000,
        tbl_formatting='MARKDOWN',
        tbl_cols=MAX_N_COLUMNS):
        r_df_str = str(r_df.select(r_df.columns[:MAX_N_COLUMNS]).sample(N_ROWS_SAMPLE, with_replacement=True))
        s_df_str = str(s_df.select(s_df.columns[:MAX_N_COLUMNS]).sample(N_ROWS_SAMPLE, with_replacement=True))

    prompt = (
        f"{i}/{UP_TO_ROW} - Pair {r_pkg_id} - {s_pkg_id}\n"
        "Consider the following information:\n"
        f"The table {r_rsc_name} is about: {r_pkg_notes}.\n"
        f"Some keywords and tags about it are: {r_pkg_keywords}, {r_pkg_tags}.\n"
        f"Example rows:\n{r_df_str}"
        f"\n{'-' * 300}\n\n"
        f"The table {s_rsc_name} is about: {s_pkg_notes}.\n"
        f"Some keywords and tags about it are: {s_pkg_keywords}, {s_pkg_tags}.\n"
        f"Example rows:\n{s_df_str}"
        f"\n{'-' * 300}\n\n"
        f"The columns that joins are {r_col_name=}, {s_col_name=}, \n"
        f"and some of the common values in these columns are: {common_cells}.\n"
        "Define a relationship quality score for the two tables. "
        "The join columns may not be identical or may not perfect6ly align. "
        "Focus on the meaningfulness of the operation between the given tables, "
        "base your choice on their description and keywords."
    )
    while True:
        print(prompt)
        try:
            human_score = int(input('Give the score: '))
        except:
            clear_output(wait=True)
            continue
        break
    clear_output(wait=True)

    evaluations.append([score, human_score])

99/100 - Pair dc1b664c-5c48-44c9-b69f-2544bda76f6b - 08cc85f3-ecad-4a97-accb-6e8509c5e157
Consider the following information:
The table  is about: This dataset provides a list of the tests undertaken by APHA testing laboratories on pig samples in 2006. The dataset includes the following fields: Year; Species class; Species; Test code; Test description; Number of tests (the volume of tests performed in the 12 month period). Attribution statemen.
Some keywords and tags about it are: [], [{'display_name': 'environment', 'id': 'aaa2ee64-885c-4ff1-b881-c097ee97eda8', 'name': 'environment', 'state': 'active', 'vocabulary_id': None}, {'display_name': 'laboratory test', 'id': '4fab3e15-ff0e-4939-aa35-f5e1634e74a2', 'name': 'laboratory test', 'state': 'active', 'vocabulary_id': None}].
Example rows:
| test_code | year_test_approved | species_class | species | test_description           | no._tests |
| ---       | ---                | ---           | ---     | ---                        | ---   

In [20]:
import statistics

differences = list(map(lambda p: p[1] - p[0], filter(lambda e: e[0] >= 0, evaluations)))

print(f"Total evaluated cases: {len(differences)}")
print(f"Mean difference to agent-based evaluation: {round(statistics.mean(differences), 3)}"), 
print(f"Relative stdev: {round(statistics.stdev(differences), 3)}")

same_score = sum(d == 0 for d in differences)
print(f"Same-score cases: {same_score} ({(same_score * 100 // len(differences))}%)")

Total evaluated cases: 100
Mean difference to agent-based evaluation: -0.43
Relative stdev: 1.451
Same-score cases: 29 (29%)


In [22]:
import matplotlib.pyplot as plt
from collections import Counter

#plt.plot(sorted(differences))
Counter(differences)

Counter({0: 29, 1: 25, -1: 23, -2: 16, 2: 3, -3: 2, -6: 1, -7: 1})